# 04_glove_fasttext: FastText Subwords on Gutenberg Corpus
    
This notebook trains FastText and Word2Vec models on Gutenberg's *Alice in Wonderland* to demonstrate how subword n-grams resolve Out-of-Vocabulary (OOV) queries.


In [1]:
import re
import requests
from gensim.models import FastText, Word2Vec

# 1. Load Alice in Wonderland from NLTK gutenberg corpus
import nltk
nltk.download('gutenberg', quiet=True)
from nltk.corpus import gutenberg
sentences_raw = gutenberg.sents('carroll-alice.txt')

cleaned_sentences = []
for s in sentences_raw:
    words = [w.lower() for w in s if re.match(r"^\w+$", w)]
    if 5 < len(words) < 35:
        cleaned_sentences.append(words)

train_sentences = cleaned_sentences[:500]

# 2. Train Word2Vec
w2v = Word2Vec(train_sentences, vector_size=10, window=3, min_count=2, epochs=20)

# 3. Train FastText
ft = FastText(train_sentences, vector_size=10, window=3, min_count=2, min_n=3, max_n=6, epochs=20)

print("Vocabulary keys in Word2Vec index:", list(w2v.wv.key_to_index.keys())[:10])

# 4. Attempt OOV word retrieval (e.g. 'alicean' - not in vocabulary)
try:
    vector = w2v.wv["alicean"]
except KeyError:
    print("\n[Word2Vec Error]: Word 'alicean' is out of vocabulary!")

# FastText handles the OOV word via character subword n-grams
ft_vector = ft.wv["alicean"]
print("\nFastText Vector for OOV word 'alicean':\n", ft_vector)

# Similar words lookup
print("\nFastText similarity 'alice' vs 'alicean':", ft.wv.similarity("alice", "alicean"))


Vocabulary keys in Word2Vec index: ['the', 'i', 'and', 'to', 'it', 'a', 'she', 'you', 'of', 'alice']

[Word2Vec Error]: Word 'alicean' is out of vocabulary!

FastText Vector for OOV word 'alicean':
 [ 0.26535594 -0.02521512  0.18308373 -0.7056187   0.19021201  0.17507018
  1.0396981   0.6853366  -0.01807771 -0.15207484]

FastText similarity 'alice' vs 'alicean': 0.9997928


### Output Explanation
- Querying standard Word2Vec with `"alicean"` crashes because the exact string is not in the training corpus.
- FastText splits `"alicean"` into character n-grams (e.g., `ali`, `lic`, `ice`, `cea`, `ean`) and sums their representations, returning a valid similarity score.
